# Kaggle Submission Notebook — Retrieval Engine Competition

**Group:** SeaFour  
**Pipeline:** Load data → Preprocess → Retrieve (BM25+ / TF-IDF / Embedding / Hybrid) → Evaluate on train → Generate submission CSV

This notebook implements four retrieval methods:
1. **TF-IDF** — sparse lexical baseline with bigrams and cosine similarity
2. **BM25+** — probabilistic lexical model with length normalisation
3. **Embedding Search** — dense semantic retrieval using Sentence-Transformers
4. **Hybrid (BM25+ → Embedding Re-ranking)** — BM25+ retrieves candidates, embedding model re-scores them, scores are fused

We evaluate every method on the training queries (MAP@100, Recall@100), then produce the final Kaggle submission CSV with the best-performing model.

In [ ]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
from pathlib import Path
import csv, json, re, os, time, hashlib, pickle

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── Data directory auto-detection (Kaggle → local fallback) ─────────────────
DATA_DIR = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    if 'docs.json' in filenames:
        DATA_DIR = Path(dirname)
        break

if DATA_DIR is None:
    DATA_DIR = Path('../data')
    if not DATA_DIR.exists():
        raise FileNotFoundError(
            "Could not find data directory. "
            "On Kaggle, add the competition dataset; "
            "locally, place files in ../data/."
        )

print(f"Data directory : {DATA_DIR}")

# ── Tuneable parameters ─────────────────────────────────────────────────────
OUTPUT_PATH     = Path('solutions_SeaFour.csv')
TOP_K           = 100
FINAL_MODEL     = 'hybrid_merge_dedup'  # 'bm25', 'tfidf', 'embedding', 'hybrid', 'hybrid_merge_dedup'
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'   # lightweight sentence-transformer
EMBEDDING_BATCH = 256

# ── Cache configuration (auto save/load) ───────────────────────────────────
CACHE_DIR             = Path('./cache')
MODEL_CACHE_DIR       = CACHE_DIR / 'sentence_transformers'
EMBEDDING_CACHE_DIR   = CACHE_DIR / 'embeddings'
TFIDF_CACHE_DIR       = CACHE_DIR / 'tfidf'
BM25_CACHE_DIR        = CACHE_DIR / 'bm25'
ENABLE_EMBEDDING_CACHE = True
ENABLE_CLASSIC_CACHE   = True

# ── Normalization & classic model parameters ────────────────────────────────
NORMALIZATION_CONFIG = {
    'lowercase': True,
    'replace_separators': True,      # replace -, _, / with spaces
    'separator_chars': '-_/',
    'collapse_whitespace': True,
    'strip': True,
}
TOKEN_PATTERN = r'[a-z0-9]+'

TFIDF_CONFIG = {
    'lowercase': True,
    'ngram_range': (1, 2),
    'min_df': 2,
}

BM25_CONFIG = {
    'k1': 1.5,
    'b': 0.75,
    'delta': 1.0,
}

for cache_path in [
    CACHE_DIR,
    MODEL_CACHE_DIR,
    EMBEDDING_CACHE_DIR,
    TFIDF_CACHE_DIR,
    BM25_CACHE_DIR,
]:
    cache_path.mkdir(parents=True, exist_ok=True)

# ── Hybrid-specific parameters ──────────────────────────────────────────────
HYBRID_BM25_CANDIDATES = 500      # BM25+ retrieves this many candidates per query
HYBRID_ALPHA           = 0.35     # weight for BM25+ score in fusion (1-α for embedding)

# ── Hybrid merge+dedup-specific parameters ───────────────────────────────
HYBRID_MERGE_BM25_CANDIDATES  = 700   # lexical candidate pool size
HYBRID_MERGE_DENSE_CANDIDATES = 700   # dense candidate pool size
HYBRID_MERGE_RRF_K            = 60    # RRF constant (typical range: 10..100)


In [ ]:
# ── 2. Shared Preprocessing ──────────────────────────────────────────────────

def value_to_text(value):
    """Normalise None / NaN / list / tuple → string."""
    if value is None:
        return ''
    if isinstance(value, (list, tuple)):
        return ' '.join(str(v) for v in value)
    if pd.isna(value):
        return ''
    return str(value)


def normalize_text(text):
    """Apply configurable text normalization used across all retrievers."""
    txt = str(text or '')
    if NORMALIZATION_CONFIG.get('replace_separators', True):
        chars = NORMALIZATION_CONFIG.get('separator_chars', '-_/')
        if chars:
            txt = re.sub(f"[{re.escape(chars)}]", ' ', txt)
    if NORMALIZATION_CONFIG.get('lowercase', True):
        txt = txt.lower()
    if NORMALIZATION_CONFIG.get('collapse_whitespace', True):
        txt = re.sub(r'\s+', ' ', txt)
    if NORMALIZATION_CONFIG.get('strip', True):
        txt = txt.strip()
    return txt


def create_content_column(df, columns):
    """Build a single lowercase 'content' field from *columns*."""
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            out[col] = ''
    merged = []
    for _, row in out[columns].iterrows():
        raw_text = ' '.join(value_to_text(row[col]) for col in columns)
        text = normalize_text(raw_text)
        merged.append(text)
    out['content'] = merged
    out['id'] = out['id'].astype(str)
    return out


_TOKEN_RE = re.compile(TOKEN_PATTERN)

def tokenize(text):
    """Tokeniser aligned with configured normalization."""
    txt = normalize_text(text)
    return _TOKEN_RE.findall(txt)


In [ ]:
# ── 3. Retrieval Methods ─────────────────────────────────────────────────────

# ── Cache Helpers (auto save/load) ───────────────────────────────────────────
_MODEL_MEMORY_CACHE = {}
_ARRAY_MEMORY_CACHE = {}
_OBJECT_MEMORY_CACHE = {}


def _safe_component(value):
    return re.sub(r'[^A-Za-z0-9._-]+', '_', str(value))


def _hash_payload(payload):
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=True, default=str)
    return hashlib.sha1(raw.encode('utf-8')).hexdigest()[:16]


def _normalization_signature():
    payload = {
        'normalization': NORMALIZATION_CONFIG,
        'token_pattern': TOKEN_PATTERN,
    }
    return _hash_payload(payload)


def _dataframe_fingerprint(df, columns):
    h = hashlib.sha1()
    h.update(str(len(df)).encode('utf-8'))
    for col in columns:
        h.update(col.encode('utf-8'))
        hashed = pd.util.hash_pandas_object(df[col].astype(str), index=False).values
        h.update(hashed.tobytes())
    return h.hexdigest()[:16]


def _load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


def _save_pickle(path, obj):
    with open(path, 'wb') as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


def _load_sentence_model(model_name):
    from sentence_transformers import SentenceTransformer

    if model_name in _MODEL_MEMORY_CACHE:
        return _MODEL_MEMORY_CACHE[model_name]

    safe_model_name = _safe_component(model_name)
    local_model_dir = MODEL_CACHE_DIR / safe_model_name
    if local_model_dir.exists():
        print(f"  Loading model weights from cache: {local_model_dir}")
        model = SentenceTransformer(str(local_model_dir))
    else:
        print(f"  Downloading model weights: {model_name}")
        model = SentenceTransformer(model_name)
        local_model_dir.mkdir(parents=True, exist_ok=True)
        model.save(str(local_model_dir))
        print(f"  Saved model weights to cache: {local_model_dir}")

    _MODEL_MEMORY_CACHE[model_name] = model
    return model


def _load_or_encode_embeddings(df, kind, model, model_name, batch_size):
    signature = _dataframe_fingerprint(df, ['id', 'content'])
    norm_signature = _normalization_signature()
    cache_name = f"{kind}_{_safe_component(model_name)}_{norm_signature}_{signature}.npy"
    cache_path = EMBEDDING_CACHE_DIR / cache_name
    memory_key = str(cache_path.resolve())

    if memory_key in _ARRAY_MEMORY_CACHE:
        return _ARRAY_MEMORY_CACHE[memory_key]

    if ENABLE_EMBEDDING_CACHE and cache_path.exists():
        print(f"  Loading {kind} embeddings from cache: {cache_path.name}")
        embeddings = np.load(cache_path)
    else:
        print(f"  Encoding {len(df)} {kind} …")
        embeddings = model.encode(
            df['content'].tolist(),
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        embeddings = np.asarray(embeddings, dtype=np.float32)
        if ENABLE_EMBEDDING_CACHE:
            np.save(cache_path, embeddings)
            print(f"  Saved {kind} embeddings to cache: {cache_path.name}")

    _ARRAY_MEMORY_CACHE[memory_key] = embeddings
    return embeddings


def _tfidf_param_candidates():
    candidates = [dict(TFIDF_CONFIG)]
    min_df = TFIDF_CONFIG.get('min_df', 1)
    if isinstance(min_df, int) and min_df > 1:
        fallback = dict(TFIDF_CONFIG)
        fallback['min_df'] = 1
        candidates.append(fallback)
    return candidates


def _build_or_load_tfidf_artifacts(docs_df):
    docs_signature = _dataframe_fingerprint(docs_df, ['id', 'content'])
    norm_signature = _normalization_signature()
    doc_ids = docs_df['id'].to_numpy()

    for params in _tfidf_param_candidates():
        cache_key = _hash_payload({
            'docs_signature': docs_signature,
            'normalization_signature': norm_signature,
            'tfidf_params': params,
        })
        cache_path = TFIDF_CACHE_DIR / f"tfidf_{cache_key}.pkl"
        memory_key = str(cache_path.resolve())

        if memory_key in _OBJECT_MEMORY_CACHE:
            return _OBJECT_MEMORY_CACHE[memory_key]

        if ENABLE_CLASSIC_CACHE and cache_path.exists():
            print(f"  Loading TF-IDF artifacts from cache: {cache_path.name}")
            artifacts = _load_pickle(cache_path)
            _OBJECT_MEMORY_CACHE[memory_key] = artifacts
            return artifacts

    params = dict(TFIDF_CONFIG)
    vectorizer = TfidfVectorizer(**params)
    try:
        doc_vectors = vectorizer.fit_transform(docs_df['content'])
    except ValueError as err:
        if 'After pruning, no terms remain' not in str(err):
            raise
        if params.get('min_df', 1) == 1:
            raise
        params['min_df'] = 1
        vectorizer = TfidfVectorizer(**params)
        doc_vectors = vectorizer.fit_transform(docs_df['content'])

    cache_key = _hash_payload({
        'docs_signature': docs_signature,
        'normalization_signature': norm_signature,
        'tfidf_params': params,
    })
    cache_path = TFIDF_CACHE_DIR / f"tfidf_{cache_key}.pkl"
    memory_key = str(cache_path.resolve())
    artifacts = {
        'vectorizer': vectorizer,
        'doc_vectors': doc_vectors,
        'doc_ids': doc_ids,
        'params': params,
    }

    if ENABLE_CLASSIC_CACHE:
        _save_pickle(cache_path, artifacts)
        print(f"  Saved TF-IDF artifacts to cache: {cache_path.name}")

    _OBJECT_MEMORY_CACHE[memory_key] = artifacts
    return artifacts


def _build_or_load_bm25_index(docs_df):
    from rank_bm25 import BM25Plus

    docs_signature = _dataframe_fingerprint(docs_df, ['id', 'content'])
    norm_signature = _normalization_signature()
    cache_key = _hash_payload({
        'docs_signature': docs_signature,
        'normalization_signature': norm_signature,
        'bm25_params': BM25_CONFIG,
    })
    cache_path = BM25_CACHE_DIR / f"bm25_{cache_key}.pkl"
    memory_key = str(cache_path.resolve())

    if memory_key in _OBJECT_MEMORY_CACHE:
        return _OBJECT_MEMORY_CACHE[memory_key]

    if ENABLE_CLASSIC_CACHE and cache_path.exists():
        print(f"  Loading BM25 index from cache: {cache_path.name}")
        artifacts = _load_pickle(cache_path)
        _OBJECT_MEMORY_CACHE[memory_key] = artifacts
        return artifacts

    tokenized_corpus = [tokenize(text) for text in docs_df['content']]
    bm25 = BM25Plus(tokenized_corpus, **BM25_CONFIG)
    artifacts = {
        'bm25': bm25,
        'doc_ids': docs_df['id'].to_numpy(),
    }

    if ENABLE_CLASSIC_CACHE:
        _save_pickle(cache_path, artifacts)
        print(f"  Saved BM25 index to cache: {cache_path.name}")

    _OBJECT_MEMORY_CACHE[memory_key] = artifacts
    return artifacts


# ── 3a. TF-IDF ──────────────────────────────────────────────────────────────
def run_tfidf_search(docs_df, queries_df, top_k=100):
    """TF-IDF with unigram+bigram features and cosine similarity."""
    top_k = min(top_k, len(docs_df))
    artifacts = _build_or_load_tfidf_artifacts(docs_df)
    vectorizer = artifacts['vectorizer']
    doc_vectors = artifacts['doc_vectors']
    doc_ids = artifacts['doc_ids']
    query_vectors = vectorizer.transform(queries_df['content'])
    scores = cosine_similarity(query_vectors, doc_vectors)

    results = []
    for i, row_scores in enumerate(scores):
        top_idx = np.argsort(row_scores)[-top_k:][::-1]
        results.append({
            'query_id': queries_df.iloc[i]['id'],
            'relevant_docs': doc_ids[top_idx].tolist(),
        })
    return results


# ── 3b. BM25+ ───────────────────────────────────────────────────────────────
def run_bm25_search(docs_df, queries_df, top_k=100):
    """BM25+ over tokenised content."""
    top_k = min(top_k, len(docs_df))
    artifacts = _build_or_load_bm25_index(docs_df)
    bm25 = artifacts['bm25']
    doc_ids = artifacts['doc_ids']

    results = []
    for _, row in queries_df.iterrows():
        query_tokens = tokenize(row['content'])
        scores = bm25.get_scores(query_tokens)
        top_idx = np.argsort(scores)[-top_k:][::-1]
        results.append({
            'query_id': row['id'],
            'relevant_docs': doc_ids[top_idx].tolist(),
        })
    return results


# ── 3c. Embedding Search ────────────────────────────────────────────────────
def run_embedding_search(docs_df, queries_df, top_k=100,
                         model_name=EMBEDDING_MODEL,
                         batch_size=EMBEDDING_BATCH):
    """Dense retrieval using a Sentence-Transformer model."""
    top_k = min(top_k, len(docs_df))
    model = _load_sentence_model(model_name)
    doc_embeddings = _load_or_encode_embeddings(
        docs_df, 'docs', model, model_name, batch_size
    )
    query_embeddings = _load_or_encode_embeddings(
        queries_df, 'queries', model, model_name, batch_size
    )

    # cosine similarity (vectors are already L2-normalised → dot product)
    scores = query_embeddings @ doc_embeddings.T
    doc_ids = docs_df['id'].to_numpy()

    results = []
    for i, row_scores in enumerate(scores):
        top_idx = np.argsort(row_scores)[-top_k:][::-1]
        results.append({
            'query_id': queries_df.iloc[i]['id'],
            'relevant_docs': doc_ids[top_idx].tolist(),
        })
    return results


# ── 3d. Hybrid: BM25+ → Embedding Re-ranking ────────────────────────────────
def run_hybrid_search(docs_df, queries_df, top_k=100,
                      bm25_candidates=HYBRID_BM25_CANDIDATES,
                      alpha=HYBRID_ALPHA,
                      model_name=EMBEDDING_MODEL,
                      batch_size=EMBEDDING_BATCH):
    """
    Two-stage hybrid retrieval:
      1. BM25+ retrieves top-N candidates (fast, lexical recall)
      2. Embedding model re-scores those candidates (semantic precision)
      3. Final score = α·norm(BM25+) + (1-α)·cos_emb
    """
    top_k = min(top_k, len(docs_df))
    bm25_candidates = max(bm25_candidates, top_k)

    # ── Stage 1: BM25+ candidate retrieval ──
    print(f"  Stage 1: BM25+ → top {bm25_candidates} candidates per query …")
    bm25_artifacts = _build_or_load_bm25_index(docs_df)
    bm25 = bm25_artifacts['bm25']
    doc_ids = bm25_artifacts['doc_ids']

    # Collect per-query BM25 candidates and scores
    bm25_candidate_indices = []   # list of arrays, one per query
    bm25_candidate_scores = []
    for _, row in queries_df.iterrows():
        query_tokens = tokenize(row['content'])
        scores = bm25.get_scores(query_tokens)
        top_idx = np.argsort(scores)[-bm25_candidates:][::-1]
        bm25_candidate_indices.append(top_idx)
        bm25_candidate_scores.append(scores[top_idx])

    # ── Stage 2: Embedding re-scoring (cached doc/query embeddings) ──
    model = _load_sentence_model(model_name)
    doc_embeddings = _load_or_encode_embeddings(
        docs_df, 'docs', model, model_name, batch_size
    )
    query_embeddings = _load_or_encode_embeddings(
        queries_df, 'queries', model, model_name, batch_size
    )

    # ── Stage 3: Score fusion ──
    results = []
    for i in range(len(queries_df)):
        cand_idx = bm25_candidate_indices[i]
        bm25_scores = bm25_candidate_scores[i]

        # Normalise BM25 scores to [0, 1]
        bm25_min = bm25_scores.min()
        bm25_max = bm25_scores.max()
        if bm25_max > bm25_min:
            bm25_norm = (bm25_scores - bm25_min) / (bm25_max - bm25_min)
        else:
            bm25_norm = np.ones_like(bm25_scores)

        # Compute embedding similarity for candidates
        cand_embs = doc_embeddings[cand_idx]
        emb_scores = query_embeddings[i] @ cand_embs.T  # dot product (normalised)

        # Fuse: α·BM25_norm + (1-α)·emb_sim
        fused = alpha * bm25_norm + (1 - alpha) * emb_scores

        # Re-rank by fused score and keep top_k
        rerank_order = np.argsort(fused)[-top_k:][::-1]
        results.append({
            'query_id': queries_df.iloc[i]['id'],
            'relevant_docs': doc_ids[cand_idx[rerank_order]].tolist(),
        })
    return results




# ── 3e. Hybrid Merge + Dedup + RRF (recall-oriented) ───────────────────────
def run_hybrid_merge_dedup_search(docs_df, queries_df, top_k=100,
                                  bm25_candidates=HYBRID_MERGE_BM25_CANDIDATES,
                                  dense_candidates=HYBRID_MERGE_DENSE_CANDIDATES,
                                  rrf_k=HYBRID_MERGE_RRF_K,
                                  model_name=EMBEDDING_MODEL,
                                  batch_size=EMBEDDING_BATCH):
    """
    Recall-oriented hybrid retrieval:
      1. Retrieve BM25 top-N and dense top-M independently
      2. Merge candidate lists and deduplicate document ids
      3. Rank merged candidates with Reciprocal Rank Fusion (RRF)
    """
    top_k = min(top_k, len(docs_df))
    bm25_candidates = max(bm25_candidates, top_k)
    dense_candidates = max(dense_candidates, top_k)

    # BM25 index
    bm25_artifacts = _build_or_load_bm25_index(docs_df)
    bm25 = bm25_artifacts['bm25']
    doc_ids = bm25_artifacts['doc_ids']

    # Dense index (single encoding pass for all docs, with cache)
    model = _load_sentence_model(model_name)
    doc_embeddings = _load_or_encode_embeddings(
        docs_df, 'docs', model, model_name, batch_size
    )
    query_embeddings = _load_or_encode_embeddings(
        queries_df, 'queries', model, model_name, batch_size
    )
    dense_scores_all = query_embeddings @ doc_embeddings.T

    results = []
    for i, (_, row) in enumerate(queries_df.iterrows()):
        bm25_scores = bm25.get_scores(tokenize(row['content']))
        bm25_top = np.argsort(bm25_scores)[-bm25_candidates:][::-1]
        dense_top = np.argsort(dense_scores_all[i])[-dense_candidates:][::-1]

        # Merge then deduplicate to maximize candidate coverage
        merged = np.concatenate([bm25_top, dense_top])
        seen = set()
        dedup_indices = []
        for idx in merged:
            j = int(idx)
            if j not in seen:
                seen.add(j)
                dedup_indices.append(j)

        # Reciprocal Rank Fusion over merged unique candidates
        bm25_rank = {int(idx): rank for rank, idx in enumerate(bm25_top, start=1)}
        dense_rank = {int(idx): rank for rank, idx in enumerate(dense_top, start=1)}

        fused_scores = []
        for idx in dedup_indices:
            score = 0.0
            b_rank = bm25_rank.get(idx)
            d_rank = dense_rank.get(idx)
            if b_rank is not None:
                score += 1.0 / (rrf_k + b_rank)
            if d_rank is not None:
                score += 1.0 / (rrf_k + d_rank)
            fused_scores.append(score)

        rerank_order = np.argsort(fused_scores)[-top_k:][::-1]
        ranked_doc_ids = [doc_ids[dedup_indices[pos]] for pos in rerank_order]

        results.append({
            'query_id': row['id'],
            'relevant_docs': ranked_doc_ids,
        })

    return results


# ── Dispatcher ───────────────────────────────────────────────────────────────
MODELS = {
    'tfidf':              run_tfidf_search,
    'bm25':               run_bm25_search,
    'embedding':          run_embedding_search,
    'hybrid':             run_hybrid_search,
    'hybrid_merge_dedup': run_hybrid_merge_dedup_search,
}

def run_retrieval(model_name, docs_df, queries_df, top_k=100):
    if model_name not in MODELS:
        raise ValueError(f"Unknown model '{model_name}'. Choose from {list(MODELS)}")
    print(f"Running {model_name} …")
    t0 = time.time()
    results = MODELS[model_name](docs_df, queries_df, top_k=top_k)
    print(f"  Done in {time.time() - t0:.1f}s")
    return results


In [ ]:
# ── 4. Evaluation Helpers ────────────────────────────────────────────────────

def load_ground_truth(path):
    """Load qgts_train.json → {query_id: set(doc_ids)}."""
    with open(path, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    gt = {}
    for qid, info in raw.items():
        gt[str(qid)] = {str(d['doc_id']) for d in info['relevant_doc_ids']}
    return gt


def mean_average_precision(results, ground_truth, k=100):
    """Compute MAP@K over results that have a matching ground-truth entry."""
    aps = []
    for item in results:
        qid = str(item['query_id'])
        if qid not in ground_truth:
            continue
        relevant = ground_truth[qid]
        hits = 0
        score = 0.0
        for rank, doc_id in enumerate(item['relevant_docs'][:k], 1):
            if str(doc_id) in relevant:
                hits += 1
                score += hits / rank
        ap = score / len(relevant) if relevant else 0.0
        aps.append(ap)
    return np.mean(aps) if aps else 0.0


def recall_at_k(results, ground_truth, k=100):
    """Compute mean Recall@K."""
    recalls = []
    for item in results:
        qid = str(item['query_id'])
        if qid not in ground_truth:
            continue
        relevant = ground_truth[qid]
        retrieved = {str(d) for d in item['relevant_docs'][:k]}
        recalls.append(len(relevant & retrieved) / len(relevant) if relevant else 0.0)
    return np.mean(recalls) if recalls else 0.0


# ── 5. Kaggle CSV Writer ────────────────────────────────────────────────────

def write_kaggle_submission(results, sample_csv_path, output_csv_path):
    """Write results in exact Kaggle submission format."""
    pred_map = {
        str(item['query_id']): [str(d) for d in item['relevant_docs']]
        for item in results
    }
    with open(sample_csv_path, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)
    if fieldnames is None or len(fieldnames) < 2:
        raise ValueError('Invalid sample submission format.')
    id_col = fieldnames[0]
    pred_col = fieldnames[1]
    category_col = fieldnames[2] if len(fieldnames) >= 3 else None

    with open(output_csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            qid = str(row[id_col])
            if qid not in pred_map:
                raise ValueError(f'Missing prediction for query_id: {qid}')
            out_row = {id_col: qid, pred_col: json.dumps(pred_map[qid])}
            if category_col is not None:
                out_row[category_col] = row.get(category_col, '?') or '?'
            writer.writerow(out_row)

In [ ]:
# ── 6. Load & Preprocess ─────────────────────────────────────────────────────
docs_df         = pd.read_json(DATA_DIR / 'docs.json')
test_queries_df = pd.read_json(DATA_DIR / 'queries_test.json')
train_queries_df = pd.read_json(DATA_DIR / 'queries_train.json')
sample_submission_path = DATA_DIR / 'submission.csv'
ground_truth = load_ground_truth(DATA_DIR / 'qgts_train.json')

docs_df          = create_content_column(docs_df, ['title', 'text', 'tags'])
test_queries_df  = create_content_column(test_queries_df, ['title', 'text'])
train_queries_df = create_content_column(train_queries_df, ['title', 'text'])

print(f"Documents : {len(docs_df):,}")
print(f"Train queries : {len(train_queries_df):,}")
print(f"Test queries  : {len(test_queries_df):,}")
print(f"Ground truth  : {len(ground_truth):,} queries")

## Evaluation on Training Queries

We evaluate all four retrieval methods on the 327 training queries using **MAP@100** and **Recall@100**, then choose the best model for the final test submission.

In [ ]:
!pip install rank_bm25

In [ ]:
# ── 7. Evaluate All Models on Train Queries ──────────────────────────────────
eval_results = {}

for model_name in ["embedding", "hybrid_merge_dedup"]: #['tfidf', 'bm25', 'embedding', 'hybrid', 'hybrid_merge_dedup']:
    results = run_retrieval(model_name, docs_df, train_queries_df, top_k=TOP_K)
    m = mean_average_precision(results, ground_truth, k=TOP_K)
    r = recall_at_k(results, ground_truth, k=TOP_K)
    eval_results[model_name] = {'MAP@100': m, 'Recall@100': r}
    print(f"  {model_name:12s}  MAP@100 = {m:.4f}   Recall@100 = {r:.4f}")

eval_df = pd.DataFrame(eval_results).T
eval_df.index.name = 'Model'
print("\n─── Summary ───")
eval_df


## Generate Test Submission

Using the model selected by `FINAL_MODEL`, we retrieve documents for the 141 test queries and write the Kaggle submission CSV.

In [ ]:
# ── 8. Generate Final Submission ──────────────────────────────────────────────
test_results = run_retrieval(FINAL_MODEL, docs_df, test_queries_df, top_k=TOP_K)
write_kaggle_submission(test_results, sample_submission_path, OUTPUT_PATH)
print(f"Saved: {OUTPUT_PATH.resolve()}")

In [ ]:
# ── 9. Preview Submission ─────────────────────────────────────────────────────
submission_preview = pd.read_csv(OUTPUT_PATH)
print(f"Rows: {len(submission_preview)}, Columns: {list(submission_preview.columns)}")
submission_preview.head()